# Wind & Solar 5-Minute Generation — Endpoint Check

Pre-scraper review notebook. Run this **before** building `ercot_wpp_5min.py` / `ercot_spp_5min.py`
so the human confirms endpoints, params, history depth, and expected file size.

| Product | Description | 
|---|---|
| NP4-743-CD | Wind PP — actual 5-min avg by geo | 
| NP4-746-CD | Solar PP — actual 5-min avg by geo | 
| NP4-733-CD | Wind PP — actual 5-min avg, system-wide | 
| NP4-738-CD | Solar PP — actual 5-min avg, system-wide | 
| NP4-745-CD | Solar PP — hourly by geo | 

Setup: `cp .env.example .env`, fill ERCOT credentials, then Run All.

**Phase 1 — Learn:** endpoint URL, field schema, unfiltered sample.

**Phase 2 — History depth:** earliest date the live API returns.

**Phase 3 — Archive depth:** earliest doc in the archive API (backfill route).

**Phase 4 — Volume:** observed row counts → parquet size estimate.

---
### Reference — zone column names in existing parquets

| Geography | Names as they appear in data |
|---|---|
| **Weather zones (8)** | `Coast, East, FarWest, NCent, North, SCent, South, West` |
| **Load zones (8) + hubs (7)** | `LZ_{NORTH,SOUTH,WEST,HOUSTON,AEN,CPS,LCRA,RAYBN}`, `HB_{NORTH,SOUTH,WEST,HOUSTON,PAN,BUSAVG,HUBAVG}` |
| **Forecast zones (4)** | `north, south, west, houston` |
| **Wind regions (5)** | `genPanhandle, genCoastal, genSouth, genWest, genNorth` |
| **Solar regions (6)** | `genCenterWest, genNorthWest, genFarWest, genFarEast, genSouthEast, genCenterEast` |

⚠️ Same word ≠ same zone across systems (`north` appears in 4 of the 5).

In [9]:
import os, time
import requests
import pandas as pd
from datetime import date, timedelta
from dotenv import load_dotenv

load_dotenv()

_TOKEN_URL = (
    'https://ercotb2c.b2clogin.com'
    '/ercotb2c.onmicrosoft.com'
    '/B2C_1_PUBAPI-ROPC-FLOW'
    '/oauth2/v2.0/token'
)
_CLIENT_ID   = 'fec253ea-0d06-4272-a5e6-b478baeecd70'
_token_cache = {'token': None, 'expires_at': 0.0}

def _get_token() -> str:
    if _token_cache['token'] and time.time() < _token_cache['expires_at']:
        return _token_cache['token']
    resp = requests.post(_TOKEN_URL, data={
        'username'     : os.environ['ERCOT_USERNAME'],
        'password'     : os.environ['ERCOT_PASSWORD'],
        'grant_type'   : 'password',
        'client_id'    : _CLIENT_ID,
        'scope'        : f'openid {_CLIENT_ID} offline_access',
        'response_type': 'id_token',
    }, timeout=(10, 60))
    resp.raise_for_status()
    body = resp.json()
    _token_cache['token']      = body['id_token']
    _token_cache['expires_at'] = time.time() + int(body.get('expires_in', 3600)) - 60
    return _token_cache['token']

def _hdrs() -> dict:
    return {
        'Authorization'            : f'Bearer {_get_token()}',
        'Ocp-Apim-Subscription-Key': os.environ['ERCOT_SUBSCRIPTION_KEY'],
        'Accept'                   : 'application/json',
    }

def _get(url, **kwargs):
    r = requests.get(url, headers=_hdrs(), timeout=(10, 120), **kwargs)
    r.raise_for_status()
    return r.json()

def _to_df(body: dict) -> pd.DataFrame:
    cols = [f['name'] if isinstance(f, dict) else f for f in body.get('fields', [])]
    return pd.DataFrame([dict(zip(cols, row)) for row in body.get('data', [])])

print('Auth OK —', _get_token()[:20], '...')

Auth OK — eyJhbGciOiJSUzI1NiIs ...


In [10]:
# ── PARAMETERS ─────────────────────────────────────────────────────────────────
PRODUCTS = ['NP4-743-CD', 'NP4-746-CD', 'NP4-733-CD', 'NP4-738-CD', 'NP4-745-CD']
API_BASE = 'https://api.ercot.com/api/public-reports'
# ──────────────────────────────────────────────────────────────────────────────

---
## Phase 1 — Endpoint & schema discovery

For each product: resolve the live endpoint from the product's artifact list, show every field
(`hasRange=True` ⇒ supports `XxxFrom`/`XxxTo` query params), and pull a 5-row unfiltered sample
(API defaults to today's window).

In [11]:
info = {}  # product -> {endpoint, fields_df, ts_field}

for product in PRODUCTS:
    print('=' * 100)
    try:
        meta_body = _get(f'{API_BASE}/{product.lower()}')
        artifacts = meta_body.get('artifacts', [])
        if not artifacts:
            print(f'{product}: NO ARTIFACTS — product may not be on the public API')
            continue
        if len(artifacts) > 1:
            print(f'{product}: multiple sub-reports, using first:')
            for a in artifacts:
                print(f'   {a["_links"]["endpoint"]["href"]}  —  {a["displayName"]}')
        artifact = artifacts[0]
        endpoint = artifact['_links']['endpoint']['href']

        schema    = _get(endpoint)
        fields_df = pd.DataFrame([{
            'name'      : f['name'],
            'label'     : f['label'],
            'dataType'  : f['dataType'],
            'searchable': f.get('searchable', False),
            'hasRange'  : f.get('hasRange', False),
        } for f in schema['fields']])

        # best timestamp field for range filtering (used in later phases)
        ranged = fields_df[fields_df.hasRange]
        pref   = [n for n in ranged.name
                  if any(k in n.lower() for k in ('intervalending', 'deliverydate', 'operatingday'))]
        ts_field = pref[0] if pref else (ranged.name.iloc[0] if len(ranged) else None)

        info[product] = {'endpoint': endpoint, 'fields_df': fields_df, 'ts_field': ts_field}

        print(f'{product} — {artifact["displayName"]}')
        print(f'Endpoint       : {endpoint}')
        print(f'Range field    : {ts_field}  (→ params {ts_field}From / {ts_field}To)')
        display(fields_df)

        sample_body = _get(endpoint, params={'page': 1, 'size': 5})
        smeta = sample_body.get('_meta', {})
        print(f'Default window : {smeta.get("query", {}).get("parameters")}')
        print(f'totalRecords   : {smeta.get("totalRecords")}')
        display(_to_df(sample_body))
    except requests.HTTPError as e:
        print(f'{product}: HTTP error — {e}')

NP4-743-CD — Wind Power Production - Actual 5-Minute Averaged Values by Geographical Region
Endpoint       : https://api.ercot.com/api/public-reports/np4-743-cd/wpp_actual_5min_avg_values_geo
Range field    : intervalEnding  (→ params intervalEndingFrom / intervalEndingTo)


,name,label,dataType,searchable,hasRange
0,postedDatetime,Posted,DATETIME,True,True
1,intervalEnding,Interval Ending,DATETIME,True,True
2,genSystemWide,Generation System Wide,DOUBLE,True,True
3,panhandle,Panhandle,DOUBLE,True,True
4,coastal,Coastal,DOUBLE,True,True
5,south,South,DOUBLE,True,True
6,west,West,DOUBLE,True,True
7,north,North,DOUBLE,True,True
8,HSLSystemWide,HSL System Wide,DOUBLE,True,True
9,DSTFlag,DST Flag,BOOLEAN,True,False


Default window : {}
totalRecords   : 3235457


,postedDatetime,intervalEnding,genSystemWide,panhandle,coastal,south,west,north,HSLSystemWide,DSTFlag
0,2026-07-04T23:55:37,2026-07-04T23:50:00,17142.62,668.53,1543.65,2638.05,10607.21,1685.19,17716.05,False
1,2026-07-04T23:55:37,2026-07-04T23:45:00,17239.87,676.93,1584.44,2610.53,10663.54,1704.43,17799.56,False
2,2026-07-04T23:55:37,2026-07-04T23:40:00,17156.71,664.25,1614.71,2600.38,10588.97,1688.39,17735.52,False
3,2026-07-04T23:55:37,2026-07-04T23:35:00,17127.70,695.49,1636.23,2606.85,10568.14,1620.99,17691.47,False
4,2026-07-04T23:55:37,2026-07-04T23:30:00,17075.79,778.96,1633.53,2614.02,10514.24,1535.04,17652.17,False


NP4-746-CD — Solar Power Production - Actual 5-Minute Averaged Values by Geographical Region
Endpoint       : https://api.ercot.com/api/public-reports/np4-746-cd/spp_actual_5min_avg_values_geo
Range field    : intervalEnding  (→ params intervalEndingFrom / intervalEndingTo)


...

,postedDatetime,intervalEnding,genSystemWide,HSLSystemWide,DSTFlag
0,2026-07-04T23:55:20,2026-07-04T23:50:00,0.81,0.46,False
1,2026-07-04T23:55:20,2026-07-04T23:45:00,1.00,0.46,False
2,2026-07-04T23:55:20,2026-07-04T23:40:00,0.94,0.48,False
3,2026-07-04T23:55:20,2026-07-04T23:35:00,0.99,0.46,False
4,2026-07-04T23:55:20,2026-07-04T23:30:00,1.56,0.95,False


NP4-745-CD — Solar Power Production - Hourly Averaged Actual and Forecasted Values by Geographical Region
Endpoint       : https://api.ercot.com/api/public-reports/np4-745-cd/spp_hrly_actual_fcast_geo
Range field    : deliveryDate  (→ params deliveryDateFrom / deliveryDateTo)


,name,label,dataType,searchable,hasRange
0,postedDatetime,Posted,DATETIME,True,True
1,deliveryDate,Delivery Date,DATE,True,True
2,hourEnding,Hour Ending,INTEGER,True,True
3,genSystemWide,Generation System Wide,DOUBLE,True,True
4,COPHSLSystemWide,COP HSL System Wide,DOUBLE,True,True
5,STPPFSystemWide,STPPF System Wide,DOUBLE,True,True
6,PVGRPPSystemWide,PVGRPP System Wide,DOUBLE,True,True
7,genCenterWest,Generation Center West,DOUBLE,True,True
8,COPHSLCenterWest,COP HSL Center West,DOUBLE,True,True
9,STPPFCenterWest,STPPF Center West,DOUBLE,True,True


Default window : {}
totalRecords   : 4852266


,postedDatetime,deliveryDate,hourEnding,genSystemWide,COPHSLSystemWide,STPPFSystemWide,PVGRPPSystemWide,genCenterWest,COPHSLCenterWest,STPPFCenterWest,...,genSouthEast,COPHSLSouthEast,STPPFSouthEast,PVGRPPSouthEast,genCenterEast,COPHSLCenterEast,STPPFCenterEast,PVGRPPCenterEast,HSLSystemWide,DSTFlag
0,2026-07-04T23:55:35,2026-07-11,1,None,0.0,0.0,0.0,None,0.0,0.0,...,None,0.0,0.0,0.0,None,0.0,0.0,0.0,None,False
1,2026-07-04T23:55:35,2026-07-11,2,None,0.0,0.0,0.0,None,0.0,0.0,...,None,0.0,0.0,0.0,None,0.0,0.0,0.0,None,False
2,2026-07-04T23:55:35,2026-07-11,3,None,0.0,0.0,0.0,None,0.0,0.0,...,None,0.0,0.0,0.0,None,0.0,0.0,0.0,None,False
3,2026-07-04T23:55:35,2026-07-11,4,None,0.0,0.0,0.0,None,0.0,0.0,...,None,0.0,0.0,0.0,None,0.0,0.0,0.0,None,False
4,2026-07-04T23:55:35,2026-07-11,5,None,0.0,0.0,0.0,None,0.0,0.0,...,None,0.0,0.0,0.0,None,0.0,0.0,0.0,None,False


---
## Phase 2 — Live-API history depth

Probe `totalRecords` for one-day windows at several past dates. First date with records > 0
≈ earliest live coverage. (Wind hourly NP4-742-CD only reached Dec 2023 live; expect similar here.)

In [12]:
PROBE_DATES = ['2021-01-15', '2022-01-15', '2023-01-15', '2023-12-15', '2024-06-15', '2025-06-15']

depth_rows = []
for product, d in info.items():
    ts = d['ts_field']
    if ts is None:
        continue
    for pd_date in PROBE_DATES:
        try:
            body = _get(d['endpoint'], params={
                f'{ts}From': pd_date, f'{ts}To': pd_date, 'page': 1, 'size': 1,
            })
            n = body.get('_meta', {}).get('totalRecords', 0)
        except requests.HTTPError as e:
            n = f'ERR {e.response.status_code}'
        depth_rows.append({'product': product, 'date': pd_date, 'totalRecords': n})
        time.sleep(0.3)

depth_df = pd.DataFrame(depth_rows).pivot(index='date', columns='product', values='totalRecords')
display(depth_df)
print('First row with records > 0 per column ≈ earliest live coverage.')

product,NP4-733-CD,NP4-738-CD,NP4-743-CD,NP4-745-CD,NP4-746-CD
date,,,,,
2021-01-15,0,ERR 429,0,ERR 429,0
2022-01-15,0,ERR 429,0,ERR 429,0
2023-01-15,0,ERR 429,0,ERR 429,0
2023-12-15,ERR 429,ERR 429,12,ERR 429,0
2024-06-15,ERR 429,ERR 429,12,ERR 429,0
2025-06-15,ERR 429,ERR 429,12,ERR 429,0


First row with records > 0 per column ≈ earliest live coverage.


---
## Phase 3 — Archive-API depth

The archive API (`/archive/{PRODUCT}`) is the backfill route if the live API is shallow
(same two-step docId → ZIP flow as `ercot_wpp_archive.py`). Listing is paginated newest-first;
the last page holds the oldest doc.

In [13]:
arch_rows = []
for product in info:
    url = f'{API_BASE}/archive/{product}'
    try:
        first = _get(url, params={'page': 1, 'size': 1})
        total = first.get('_meta', {}).get('totalRecords', 0)
        pages = first.get('_meta', {}).get('totalPages', 1)
        newest = first.get('archives', [{}])[0]
        oldest = _get(url, params={'page': pages, 'size': 1}).get('archives', [{}])[0]
        arch_rows.append({
            'product'    : product,
            'totalDocs'  : total,
            'newest_post': newest.get('postDatetime'),
            'oldest_post': oldest.get('postDatetime'),
        })
    except requests.HTTPError as e:
        arch_rows.append({'product': product, 'totalDocs': f'ERR {e.response.status_code}'})
    time.sleep(0.3)

display(pd.DataFrame(arch_rows))
print('oldest_post ≈ how far back a backfill can reach.')

,product,totalDocs
0,NP4-743-CD,ERR 429
1,NP4-746-CD,ERR 429
2,NP4-733-CD,ERR 429
3,NP4-738-CD,ERR 429
4,NP4-745-CD,ERR 429


oldest_post ≈ how far back a backfill can reach.


---
## Phase 4 — Volume & parquet size estimate

Observed `totalRecords` for one full recent month, then extrapolated to a 2021-01-01 → today pull.
Rows are **pre-dedupe** (each 5-min interval appears ~12× across rolling 60-min snapshots);
deduped rows ≈ raw ÷ 12. Bytes/row benchmarks from this repo:
~50 B (snappy, deduped hourly wind) → assume ~60 B for 5-min zstd, conservatively.

In [14]:
MONTH_FROM, MONTH_TO = '2026-06-01', '2026-06-30'
BYTES_PER_ROW = 60
days_full = (date.today() - date(2021, 1, 1)).days

vol_rows = []
for product, d in info.items():
    ts = d['ts_field']
    if ts is None:
        continue
    try:
        body = _get(d['endpoint'], params={
            f'{ts}From': MONTH_FROM, f'{ts}To': MONTH_TO, 'page': 1, 'size': 1,
        })
        n_month = body.get('_meta', {}).get('totalRecords', 0)
    except requests.HTTPError:
        n_month = 0
    raw_full   = n_month / 30 * days_full
    dedup_full = raw_full / 12  # keep one snapshot per interval
    vol_rows.append({
        'product'          : product,
        'rows_june2026'    : n_month,
        'raw_rows_2021_now': f'{raw_full:,.0f}',
        'deduped_rows'     : f'{dedup_full:,.0f}',
        'est_parquet_MB'   : round(dedup_full * BYTES_PER_ROW / 1e6, 1),
    })
    time.sleep(0.3)

display(pd.DataFrame(vol_rows))

,product,rows_june2026,raw_rows_2021_now,deduped_rows,est_parquet_MB
0,NP4-743-CD,0,0,0,0.0
1,NP4-746-CD,0,0,0,0.0
2,NP4-733-CD,0,0,0,0.0
3,NP4-738-CD,0,0,0,0.0
4,NP4-745-CD,0,0,0,0.0


---
## Decision checklist (human)

- [ ] Wind product: NP4-743-CD (by geo) vs NP4-733-CD (system-wide)?
- [ ] Solar product: NP4-746-CD (by geo) vs NP4-738-CD (system-wide)?
- [ ] Confirm range-filter param names from Phase 1 (`ts_field`).
- [ ] Live depth sufficient (Phase 2), or archive backfill needed (Phase 3)?
- [ ] Estimated parquet size acceptable (Phase 4)?
- [ ] NP4-745-CD solar hourly: does live API really reach 2021? → run `ercot_spp_by_geo.py`.

Then: build `ercot_wpp_5min.py` / `ercot_spp_5min.py` (raw values kept, snapshot-deduped, zstd).